In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths to locate the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Also add parent directories of the current directory (when running from notebooks/)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 05. Information Theory - From Entropy to KL Divergence

> **Learning Goals**
> - Understand entropy $H(X)$ as a measure of "uncertainty."
> - Connect cross-entropy to the loss used in LLM training.
> - Understand how KL divergence measures differences between distributions.

## 5.1 Information and Entropy

**Information**: how surprising an event is.

$$I(x) = -\log P(x)$$

- Rare events (small $P$): high information
- Common events (large $P$): low information
- Log base 2 gives bits; log base $e$ gives nats.

**Entropy**: expected information value, or uncertainty of a distribution.

$$H(X) = -\sum_x P(x) \log P(x)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def entropy(p):
    """Entropy of a discrete distribution using natural log."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]  # Exclude zero-probability terms (0*log0 = 0)
    return -np.sum(p * np.log(p))

# Coin-flip entropy: maximum at p = 0.5
ps = np.linspace(0.01, 0.99, 100)
entropies = [entropy([p, 1-p]) for p in ps]

plt.figure(figsize=(9, 5))
plt.plot(ps, entropies, 'b-', linewidth=2)
plt.axvline(0.5, color='r', linestyle='--', label=f'Maximum entropy (p=0.5): H={entropy([0.5, 0.5]):.4f}')
plt.xlabel('P(heads) = p')
plt.ylabel('Entropy H (nat)')
plt.title('Entropy of Bernoulli(p)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch05_entropy_coin.png', dpi=100, bbox_inches='tight')
plt.show()
print("entropy is maximized at p=0.5 (completely random). It is 0 at p=0 or p=1 (deterministic).")


## 5.2 Cross-Entropy

For a true distribution $P$ and a model distribution $Q$:

$$H(P, Q) = -\sum_x P(x) \log Q(x)$$

Interpretation: average code length when data from $P$ is encoded using $Q$.

**Decomposition**:

$$H(P, Q) = H(P) + D_{\mathrm{KL}}(P \| Q)$$

- $H(P)$: inherent uncertainty of the true distribution
- $D_{\mathrm{KL}}$: mismatch between $P$ and $Q$

Therefore, minimizing cross-entropy is equivalent to minimizing KL divergence, making $Q$ closer to $P$.

In LLM training, when the target token is $y$ and the model output is $Q=\mathrm{softmax}(\mathbf{z})$:

$$\mathcal{L} = -\log Q(y)$$

This is called negative log-likelihood (NLL) or cross-entropy loss.


In [ ]:
# Cross-entropy loss implementation
def cross_entropy_loss(logits, target_idx):
    """Cross-entropy loss for one target class."""
    # softmax
    z = logits - logits.max()
    exp_z = np.exp(z)
    probs = exp_z / exp_z.sum()
    # -log p(target)
    return -np.log(probs[target_idx] + 1e-12), probs

# Example
logits = np.array([2.0, 1.0, 0.5, -0.5])
target = 0  # The label is the first token
loss, probs = cross_entropy_loss(logits, target)
print(f"Logits: {logits}")
print(f"Softmax probabilities: {probs.round(4)}")
print(f"label token index: {target}")
print(f"Cross-entropy loss: {loss:.4f}")
print(f"label probability: {probs[target]:.4f} -> -log = {-np.log(probs[target]):.4f}")

# The loss decreases as the correct logit grows
print("\nCorrect-class logit versus loss:")
for correct_logit in [0.0, 1.0, 2.0, 5.0, 10.0]:
    logits_test = np.array([correct_logit, 0.0, 0.0, 0.0])
    loss, _ = cross_entropy_loss(logits_test, 0)
    print(f"  Label logit={correct_logit:>5.1f} -> loss={loss:.4f}")


## 5.3 KL Divergence (Kullback-Leibler Divergence)

KL divergence measures the difference between two distributions $P$ and $Q$:

$$D_{\mathrm{KL}}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

Properties:
1. $D_{\mathrm{KL}}(P \| Q) \geq 0$ (non-negative)
2. $D_{\mathrm{KL}}(P \| Q) = 0$ only when $P=Q$
3. **Asymmetric**: $D_{\mathrm{KL}}(P \| Q) \neq D_{\mathrm{KL}}(Q \| P)$ in general.

Uses in LLMs:
- **RLHF**: constrain the new policy from moving too far away from the reference model.
- **Distillation**: train a smaller model to match the output distribution of a larger model.
- **DPO**: appears in preference optimization calculations.


In [ ]:
# KL divergence calculation
def kl_divergence(p, q):
    """D_KL(P || Q)."""
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    # Handle zero terms: if P(x)=0, that term contributes 0
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask] / (q[mask] + 1e-12)))

# Example distributions
P = np.array([0.5, 0.3, 0.2])
Q1 = np.array([0.5, 0.3, 0.2])  # same as P
Q2 = np.array([0.4, 0.4, 0.2])  # slightly different
Q3 = np.array([0.7, 0.2, 0.1])  # very different

print(f"P  = {P}")
print(f"Q1 = {Q1} (same as P)")
print(f"Q2 = {Q2} (slightly different)")
print(f"Q3 = {Q3} (very different)")
print()
print(f"D_KL(P || Q1) = {kl_divergence(P, Q1):.6f} (should be 0)")
print(f"D_KL(P || Q2) = {kl_divergence(P, Q2):.6f}")
print(f"D_KL(P || Q3) = {kl_divergence(P, Q3):.6f}")
print()
print("Check asymmetry:")
print(f"D_KL(P || Q3) = {kl_divergence(P, Q3):.6f}")
print(f"D_KL(Q3 || P) = {kl_divergence(Q3, P):.6f}")
print("=> The two values differ. KL divergence is not a distance.")


In [ ]:
# Visualize KL divergence: KL increases as distributions move farther apart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: KL for different Q values
ax = axes[0]
P = np.array([0.5, 0.5])
qs = np.linspace(0.01, 0.99, 100)
kls = [kl_divergence(P, [q, 1-q]) for q in qs]
ax.plot(qs, kls, 'b-', linewidth=2)
ax.axvline(0.5, color='r', linestyle='--', label='P=Q (KL=0)')
ax.set_xlabel('Q(heads)')
ax.set_ylabel('D_KL(P || Q)')
ax.set_title('KL divergence: KL increases as Q moves away from P(0.5)')
ax.legend(); ax.grid(True, alpha=0.3)

# Right: cross-entropy = H(P) + D_KL(P||Q)
ax = axes[1]
H_P = entropy(P)
ces = [H_P + kl for kl in kls]
ax.plot(qs, kls, 'b-', linewidth=2, label='D_KL(P||Q)')
ax.plot(qs, ces, 'r-', linewidth=2, label='H(P,Q) = H(P) + D_KL')
ax.axhline(H_P, color='g', linestyle='--', label=f'H(P) = {H_P:.4f}')
ax.set_xlabel('Q(heads)')
ax.set_ylabel('value')
ax.set_title('Cross-entropy = entropy + KL divergence')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/ch05_kl_divergence.png', dpi=100, bbox_inches='tight')
plt.show()


## 5.4 Perplexity (PPL)

Perplexity is a common evaluation metric for language models:

$$\mathrm{PPL} = e^{H(P, Q)}$$

It can be interpreted as the model's effective number of choices. Lower is better:

- PPL = 1: perfect prediction
- PPL = |V|: close to random prediction over the vocabulary
- Lower PPL means the model assigns higher probability to the correct tokens.


In [ ]:
# Perplexity calculation example
def perplexity(loss):
    """Compute PPL from the mean cross-entropy loss."""
    return np.exp(loss)

# Simulation: predictions for a 5-token sequence
np.random.seed(0)
seq_len = 5
vocab_size = 1000

# Compare random predictions with confident predictions
random_losses = []
good_losses = []

for t in range(seq_len):
    # Random prediction
    logits = np.random.randn(vocab_size)
    target = np.random.randint(vocab_size)
    loss, _ = cross_entropy_loss(logits, target)
    random_losses.append(loss)

    # Good prediction (boost the correct label)
    logits = np.random.randn(vocab_size) * 0.1
    target = np.random.randint(vocab_size)
    logits[target] += 5.0  # Boost correct label
    loss, _ = cross_entropy_loss(logits, target)
    good_losses.append(loss)

print(f"Random prediction:")
print(f"  Token losses: {[f'{l:.2f}' for l in random_losses]}")
print(f"  Mean loss: {np.mean(random_losses):.4f}")
print(f"  PPL: {perplexity(np.mean(random_losses)):.2f}")
print(f"  (vocabulary size {vocab_size} means closer to random)")

print(f"\nGood prediction:")
print(f"  Token losses: {[f'{l:.2f}' for l in good_losses]}")
print(f"  Mean loss: {np.mean(good_losses):.4f}")
print(f"  PPL: {perplexity(np.mean(good_losses)):.2f}")

# Relationship between PPL and loss
losses = np.linspace(0.1, 10, 100)
ppls = np.exp(losses)
plt.figure(figsize=(8, 5))
plt.plot(losses, ppls, 'b-', linewidth=2)
plt.xlabel('Cross-entropy loss')
plt.ylabel('Perplexity (PPL)')
plt.title('PPL = exp(loss)')
plt.yscale('log')
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch05_ppl.png', dpi=100, bbox_inches='tight')
plt.show()


## 5.5 Mutual Information

Mutual information measures how much knowing one variable reduces uncertainty about another:

$$I(X;Y) = \sum_{x,y} P(x,y) \log \frac{P(x,y)}{P(x)P(y)}$$

If $X$ and $Y$ are independent, $I(X;Y)=0$.


In [ ]:
# Mutual information example: two coins (independent vs correlated)
def mutual_information(joint):
    """Mutual information for a 2x2 joint distribution."""
    joint = np.asarray(joint, dtype=float)
    px = joint.sum(axis=1, keepdims=True)
    py = joint.sum(axis=0, keepdims=True)
    independent = px @ py
    mask = joint > 0
    return np.sum(joint[mask] * np.log(joint[mask] / independent[mask]))

# Two independent coins
joint_indep = np.array([[0.25, 0.25],
                        [0.25, 0.25]])
# Perfect correlation (X=Y)
joint_corr = np.array([[0.5, 0.0],
                        [0.0, 0.5]])
# Weak correlation
joint_weak = np.array([[0.4, 0.1],
                       [0.1, 0.4]])

print(f"Independent: MI = {mutual_information(joint_indep):.6f} (should be 0)")
print(f"Weak correlation: MI = {mutual_information(joint_weak):.6f}")
print(f"Perfect correlation: MI = {mutual_information(joint_corr):.6f}")


## 5.6 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Information | $I(x)=-\log P(x)$ | Surprise of an event |
| Entropy | $H(X)=-\sum P(x)\log P(x)$ | Uncertainty |
| Cross-entropy | $H(P,Q)=-\sum P(x)\log Q(x)$ | Prediction loss |
| KL divergence | $D_{\mathrm{KL}}(P\|Q)=\sum P\log(P/Q)$ | Difference between distributions |
| Perplexity | $\mathrm{PPL}=e^H$ | Effective number of choices |
| Mutual information | $I(X;Y)$ | Shared information |

### Practice Problems
1. Compute the entropy of a biased coin with $p=0.8$.
2. Compute the cross-entropy loss for a 3-class classifier with logits $[2,1,0]$ and target class 0.
3. Compute $D_{\mathrm{KL}}(P\|Q)$ for $P=[0.7,0.3]$ and $Q=[0.5,0.5]$.
4. Explain why KL divergence is not a distance metric.
5. Explain in your own words how cross-entropy, NLL, and perplexity are connected in LLM training.
